# Custom BA + 3DGS 파이프라인 (mipnerf360)

COLMAP으로 SfM 전체(Step 1-2-3)를 실행한 뒤,
**BA(Bundle Adjustment)만 직접 구현한 `bundle_adjustment.py`로 재실행**하여 결과를 비교합니다.

이후 custom BA 결과로 3DGS 학습 → 렌더링 → 평가까지 수행합니다.

In [ ]:
# === 셀 1: 설정 ===
import os, glob, shutil, struct, sys

# ========== 여기만 수정 ==========
SCENE_NAME = "bicycle"
# 사용 가능한 씬: bicycle, bonsai, counter, garden, kitchen, room, stump

IMAGE_SCALE = 4  # 이미지 스케일 (1=원본, 2=1/2, 4=1/4, 8=1/8)

RUN_COLMAP = False  # True: COLMAP 직접 실행 / False: mipnerf360 데이터셋의 COLMAP 결과 복사
DEBUG = True        # True: BA 전후 reprojection error 비교 시각화
# ================================

BASE_DIR = "/home/daeho/storage/3dgs_sba"
GS_REPO = os.path.join(BASE_DIR, "repos", "gaussian-splatting")
DATASET_DIR = os.path.join(BASE_DIR, "datasets", "mipnerf360", SCENE_NAME)

if IMAGE_SCALE == 1:
    SRC_IMAGE_DIR = os.path.join(DATASET_DIR, "images")
else:
    SRC_IMAGE_DIR = os.path.join(DATASET_DIR, f"images_{IMAGE_SCALE}")

# 작업 경로
WORK_DIR = os.path.join(BASE_DIR, "colmap_ws_sba", SCENE_NAME)
IMAGE_DIR = os.path.join(WORK_DIR, "images")
VANILLA_SPARSE = os.path.join(WORK_DIR, "sparse_vanilla", "0")  # COLMAP 원본 결과 (비교용 보존)
SPARSE_DIR = os.path.join(WORK_DIR, "sparse", "0")              # BA 결과 (3DGS가 읽는 경로)
OUTPUT_DIR = os.path.join(BASE_DIR, "output", f"{SCENE_NAME}_sba")

os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
sys.path.insert(0, BASE_DIR)

print(f"SCENE_NAME      : {SCENE_NAME}")
print(f"IMAGE_SCALE     : 1/{IMAGE_SCALE}" if IMAGE_SCALE > 1 else f"IMAGE_SCALE     : 원본")
print(f"RUN_COLMAP      : {RUN_COLMAP} ({'직접 실행' if RUN_COLMAP else '데이터셋 복사'})")
print(f"DEBUG           : {DEBUG}")
print(f"SRC_IMAGE_DIR   : {SRC_IMAGE_DIR} ({'OK' if os.path.isdir(SRC_IMAGE_DIR) else 'NOT FOUND'})")
print(f"VANILLA_SPARSE  : {VANILLA_SPARSE}")
print(f"SPARSE_DIR      : {SPARSE_DIR}")
print(f"OUTPUT_DIR      : {OUTPUT_DIR}")

src_files = sorted(glob.glob(os.path.join(SRC_IMAGE_DIR, "*")))
if src_files:
    from PIL import Image
    sample = Image.open(src_files[0])
    print(f"\n소스 이미지: {len(src_files)}장, {sample.size[0]}x{sample.size[1]}")

In [2]:
# === 셀 2: 이미지 복사 ===

os.makedirs(IMAGE_DIR, exist_ok=True)

if not os.listdir(IMAGE_DIR):
    src_files = sorted(glob.glob(os.path.join(SRC_IMAGE_DIR, "*")))
    print(f"이미지 복사 중... ({len(src_files)}장)")
    for src in src_files:
        shutil.copy2(src, os.path.join(IMAGE_DIR, os.path.basename(src)))
    print(f"✅ 복사 완료 -> {IMAGE_DIR}")
else:
    print(f"✅ 이미지 이미 존재: {len(os.listdir(IMAGE_DIR))}장")
    print("   (설정을 변경했다면 images/ 폴더를 삭제 후 다시 실행하세요)")

이미지 복사 중... (194장)
✅ 복사 완료 -> /home/daeho/storage/3dgs_sba/colmap_ws_sba/bicycle/images


In [ ]:
# === 셀 3: COLMAP 결과 준비 → sparse_vanilla/0/ 에 저장 ===
# RUN_COLMAP=True  : COLMAP을 직접 실행
# RUN_COLMAP=False : mipnerf360 데이터셋의 sparse/0/ 를 복사

colmap_files = ["cameras.bin", "images.bin", "points3D.bin"]

if RUN_COLMAP:
    # --- COLMAP 직접 실행 → 결과를 sparse_vanilla/0/ 에 저장 ---
    db_path = os.path.join(WORK_DIR, "database.db")
    tmp_sparse = os.path.join(WORK_DIR, "_tmp_sparse")

    if os.path.exists(db_path):
        os.remove(db_path)
    if os.path.exists(tmp_sparse):
        shutil.rmtree(tmp_sparse)
    os.makedirs(tmp_sparse, exist_ok=True)

    print("=== Step 1/3: Feature Extraction ===")
    !colmap feature_extractor \
        --database_path {db_path} \
        --image_path {IMAGE_DIR} \
        --ImageReader.single_camera 1 \
        --ImageReader.camera_model PINHOLE \
        --SiftExtraction.use_gpu 1

    print("\n=== Step 2/3: Exhaustive Matching ===")
    !colmap exhaustive_matcher \
        --database_path {db_path} \
        --SiftMatching.use_gpu 1

    print("\n=== Step 3/3: Mapper (COLMAP BA) ===")
    !colmap mapper \
        --database_path {db_path} \
        --image_path {IMAGE_DIR} \
        --output_path {tmp_sparse} \
        --Mapper.ba_global_function_tolerance 0.000001

    # 가장 큰 reconstruction 선택
    def count_images_bin(path):
        with open(path, "rb") as f:
            return struct.unpack("<Q", f.read(8))[0]

    sparse_dirs = sorted(glob.glob(os.path.join(tmp_sparse, "*")))
    best_dir = sparse_dirs[0]
    best_count = 0
    for sd in sparse_dirs:
        ib = os.path.join(sd, "images.bin")
        if os.path.exists(ib):
            n = count_images_bin(ib)
            print(f"  {os.path.basename(sd)}: {n} images")
            if n > best_count:
                best_count = n
                best_dir = sd

    # sparse_vanilla/0/ 에 복사
    if os.path.exists(os.path.dirname(VANILLA_SPARSE)):
        shutil.rmtree(os.path.dirname(VANILLA_SPARSE))
    os.makedirs(VANILLA_SPARSE, exist_ok=True)
    for f in colmap_files:
        shutil.copy2(os.path.join(best_dir, f), os.path.join(VANILLA_SPARSE, f))
    shutil.rmtree(tmp_sparse)

    print(f"\n✅ COLMAP 완료 → sparse_vanilla/0/ ({best_count} images)")

else:
    # --- 데이터셋에서 sparse_vanilla/0/ 로 복사 ---
    src_sparse = os.path.join(DATASET_DIR, "sparse", "0")
    if os.path.exists(os.path.dirname(VANILLA_SPARSE)):
        shutil.rmtree(os.path.dirname(VANILLA_SPARSE))
    os.makedirs(VANILLA_SPARSE, exist_ok=True)

    for f in colmap_files:
        shutil.copy2(os.path.join(src_sparse, f), os.path.join(VANILLA_SPARSE, f))

    n_imgs = struct.unpack("<Q", open(os.path.join(VANILLA_SPARSE, "images.bin"), "rb").read(8))[0]
    n_pts = struct.unpack("<Q", open(os.path.join(VANILLA_SPARSE, "points3D.bin"), "rb").read(8))[0]
    print(f"✅ mipnerf360/{SCENE_NAME}/sparse/0/ → sparse_vanilla/0/ 복사 완료")
    print(f"  Images: {n_imgs}, Points: {n_pts:,}")

In [ ]:
# === 셀 4: Custom Bundle Adjustment ===
# sparse_vanilla/0/ 를 읽어서 BA 재실행 → sparse/0/ 에 저장 (3DGS가 바로 읽는 경로)

from bundle_adjustment import BundleAdjustment

ba = BundleAdjustment(VANILLA_SPARSE)
ba.run(max_iterations=50)
ba.export(SPARSE_DIR)

In [ ]:
# === 셀 4-1: BA 전후 Reprojection Error 비교 (DEBUG=True 일 때만) ===
# sparse_vanilla/0/ (COLMAP 원본) vs sparse/0/ (custom BA 결과)
import numpy as np

if not DEBUG:
    print("DEBUG = False. 비교 시각화를 건너뜁니다.")
else:
    import matplotlib.pyplot as plt
    from bundle_adjustment_utils import (
        read_cameras_binary, read_images_binary, read_points3D_binary,
        intrinsics_to_K, qvec_to_rotmat, rotmat_to_rvec, reprojection_error,
    )

    def compute_all_reproj_errors(sparse_path):
        """sparse 경로에서 전체 reprojection error를 계산."""
        cameras = read_cameras_binary(os.path.join(sparse_path, "cameras.bin"))
        images = read_images_binary(os.path.join(sparse_path, "images.bin"))
        points3D = read_points3D_binary(os.path.join(sparse_path, "points3D.bin"))
        cam = list(cameras.values())[0]
        K = intrinsics_to_K(cam["params"])

        errors = []
        for pid, pt in points3D.items():
            xyz = pt["xyz"].reshape(1, 3)
            for img_id, kp_idx in pt["track"]:
                if img_id not in images:
                    continue
                img = images[img_id]
                R = qvec_to_rotmat(img["qvec"])
                rvec = rotmat_to_rvec(R)
                x, y, _ = img["points2D"][kp_idx]
                err = reprojection_error(xyz, np.array([[x, y]]), rvec, img["tvec"], K)
                errors.append(err[0])
        return np.array(errors)

    print("Vanilla (COLMAP) reprojection error 계산 중...")
    errors_vanilla = compute_all_reproj_errors(VANILLA_SPARSE)
    print("Custom BA reprojection error 계산 중...")
    errors_ba = compute_all_reproj_errors(SPARSE_DIR)

    # --- 시각화 ---
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # 1) 히스토그램 비교
    ax = axes[0]
    bins = np.linspace(0, min(5.0, max(errors_vanilla.max(), errors_ba.max())), 100)
    ax.hist(errors_vanilla, bins=bins, alpha=0.6, label=f"Vanilla (mean={errors_vanilla.mean():.3f})", color="steelblue")
    ax.hist(errors_ba, bins=bins, alpha=0.6, label=f"Custom BA (mean={errors_ba.mean():.3f})", color="coral")
    ax.axvline(errors_vanilla.mean(), color="steelblue", linestyle="--", lw=1.5)
    ax.axvline(errors_ba.mean(), color="coral", linestyle="--", lw=1.5)
    ax.set_xlabel("Reprojection Error (px)")
    ax.set_ylabel("Count")
    ax.set_title("Reprojection Error Distribution")
    ax.legend()

    # 2) CDF 비교
    ax = axes[1]
    for errs, label, color in [(errors_vanilla, "Vanilla", "steelblue"), (errors_ba, "Custom BA", "coral")]:
        sorted_e = np.sort(errs)
        cdf = np.arange(1, len(sorted_e) + 1) / len(sorted_e)
        ax.plot(sorted_e, cdf, label=label, color=color, lw=2)
    ax.set_xlabel("Reprojection Error (px)")
    ax.set_ylabel("CDF")
    ax.set_title("Cumulative Distribution")
    ax.set_xlim(0, 3)
    ax.legend()
    ax.grid(True, alpha=0.3)

    # 3) 통계 비교 바 차트
    ax = axes[2]
    metrics = ["Mean", "Median", "Std", "<1px (%)"]
    van_vals = [errors_vanilla.mean(), np.median(errors_vanilla), errors_vanilla.std(),
                100 * (errors_vanilla < 1.0).sum() / len(errors_vanilla)]
    ba_vals = [errors_ba.mean(), np.median(errors_ba), errors_ba.std(),
               100 * (errors_ba < 1.0).sum() / len(errors_ba)]

    x = np.arange(len(metrics))
    w = 0.35
    bars1 = ax.bar(x - w/2, van_vals, w, label="Vanilla", color="steelblue", alpha=0.8)
    bars2 = ax.bar(x + w/2, ba_vals, w, label="Custom BA", color="coral", alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(metrics)
    ax.set_title("Statistics Comparison")
    ax.legend()

    for bar, val in zip(bars1, van_vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f"{val:.2f}",
                ha="center", va="bottom", fontsize=8)
    for bar, val in zip(bars2, ba_vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f"{val:.2f}",
                ha="center", va="bottom", fontsize=8)

    plt.suptitle(f"Vanilla vs Custom BA - {SCENE_NAME} ({len(errors_vanilla):,} obs)", fontsize=14)
    plt.tight_layout()
    plt.show()

    # 텍스트 요약
    print(f"\n{'':>20} {'Vanilla':>12} {'Custom BA':>12} {'Diff':>12}")
    print("-" * 58)
    print(f"{'Mean (px)':>20} {errors_vanilla.mean():>12.4f} {errors_ba.mean():>12.4f} {errors_ba.mean()-errors_vanilla.mean():>+12.4f}")
    print(f"{'Median (px)':>20} {np.median(errors_vanilla):>12.4f} {np.median(errors_ba):>12.4f} {np.median(errors_ba)-np.median(errors_vanilla):>+12.4f}")
    print(f"{'Observations':>20} {len(errors_vanilla):>12,} {len(errors_ba):>12,}")

In [ ]:
# === 셀 5: 3DGS 학습 (custom BA 결과 사용) ===
# sparse/0/ 에 BA 결과가 반영되어 있으므로 WORK_DIR 을 바로 사용
%cd {GS_REPO}
!python train.py \
    -s {WORK_DIR} \
    -m {OUTPUT_DIR} \
    --iterations 30000 \
    --densify_grad_threshold 0.001 \
    --resolution -1 \
    --eval \
    --test_iterations 7000 15000 30000 \
    --save_iterations 7000 15000 30000

In [6]:
# === 셀 6: 렌더링 ===
%cd {GS_REPO}
!python render.py -m {OUTPUT_DIR} --iteration 30000

/home/daeho/storage/3dgs_sba/repos/gaussian-splatting
Looking for config file in /home/daeho/storage/3dgs_sba/output/bicycle_custom/cfg_args
Config file found: /home/daeho/storage/3dgs_sba/output/bicycle_custom/cfg_args
Rendering /home/daeho/storage/3dgs_sba/output/bicycle_custom
Loading trained model at iteration 30000 [14/04 16:21:27]
------------LLFF HOLD------------- [14/04 16:21:28]
Reading camera 194/194 [14/04 16:21:28]
Loading Training Cameras [14/04 16:21:28]
[ INFO ] Encountered quite large input images (>1.6K pixels width), rescaling to 1.6K.
 If this is not desired, please explicitly specify '--resolution/-r' as 1 [14/04 16:21:28]
Loading Test Cameras [14/04 16:21:47]
Rendering progress: 100%|███████████████████████| 25/25 [00:22<00:00,  1.13it/s]


In [ ]:
# === 셀 7: 메트릭 평가 (PSNR / SSIM / LPIPS) + vanilla 비교 ===
import json

%cd {GS_REPO}
!python metrics.py -m {OUTPUT_DIR}

# Custom BA 결과
results_path = os.path.join(OUTPUT_DIR, "results.json")
# Vanilla 결과 (같은 씬의 3dgs_vanilla 결과)
vanilla_path = os.path.join(BASE_DIR, "output", f"{SCENE_NAME}_vanilla", "results.json")

sba_results = None
vanilla_results = None

if os.path.exists(results_path):
    with open(results_path) as f:
        sba_results = json.load(f)

if os.path.exists(vanilla_path):
    with open(vanilla_path) as f:
        vanilla_results = json.load(f)

# 출력
if sba_results:
    print("\n=== Custom BA 결과 ===")
    for iteration, metrics in sba_results.items():
        print(f"\n[{iteration}]")
        for metric, value in metrics.items():
            print(f"  {metric}: {value:.4f}")

if vanilla_results and sba_results:
    print("\n\n=== Vanilla vs Custom BA 비교 ===")
    # 마지막 iteration 비교
    sba_key = list(sba_results.keys())[-1]
    van_key = list(vanilla_results.keys())[-1]
    sba_m = sba_results[sba_key]
    van_m = vanilla_results[van_key]

    print(f"\n{'Metric':>10} {'Vanilla':>12} {'Custom BA':>12} {'Diff':>12}")
    print("-" * 50)
    for metric in ["PSNR", "SSIM", "LPIPS"]:
        v = van_m.get(metric, 0)
        s = sba_m.get(metric, 0)
        diff = s - v
        # PSNR/SSIM: 높을수록 좋음, LPIPS: 낮을수록 좋음
        if metric == "LPIPS":
            better = "+" if diff < 0 else "-" if diff > 0 else "="
        else:
            better = "+" if diff > 0 else "-" if diff < 0 else "="
        print(f"{metric:>10} {v:>12.4f} {s:>12.4f} {diff:>+12.4f} {better}")

    if DEBUG:
        import matplotlib.pyplot as plt
        metrics = ["PSNR", "SSIM", "LPIPS"]
        van_vals = [van_m.get(m, 0) for m in metrics]
        sba_vals = [sba_m.get(m, 0) for m in metrics]

        fig, axes = plt.subplots(1, 3, figsize=(12, 4))
        for i, (metric, vv, sv) in enumerate(zip(metrics, van_vals, sba_vals)):
            ax = axes[i]
            bars = ax.bar(["Vanilla", "Custom BA"], [vv, sv],
                          color=["steelblue", "coral"], alpha=0.8)
            for bar, val in zip(bars, [vv, sv]):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                        f"{val:.4f}", ha="center", va="bottom", fontsize=10)
            ax.set_title(metric)
            ax.set_ylabel(metric)
        plt.suptitle(f"3DGS Quality: Vanilla vs Custom BA - {SCENE_NAME}", fontsize=13)
        plt.tight_layout()
        plt.show()

elif not vanilla_results:
    print(f"\n(Vanilla 결과 없음: {vanilla_path})")
    print(f"  3dgs_vanilla.ipynb 에서 {SCENE_NAME} 을 먼저 실행하면 비교 가능합니다.")

print("\n✅ 평가 완료")

In [ ]:
# === 셀 8: 렌더링 이미지 시각화 ===
import matplotlib.pyplot as plt

render_dir = os.path.join(OUTPUT_DIR, "test", "ours_30000", "renders")
gt_dir = os.path.join(OUTPUT_DIR, "test", "ours_30000", "gt")

if not os.path.isdir(render_dir):
    print(f"❌ 렌더링 결과가 없습니다: {render_dir}")
else:
    render_imgs = sorted(glob.glob(os.path.join(render_dir, "*.png")))
    gt_imgs = sorted(glob.glob(os.path.join(gt_dir, "*.png")))

    n_show = min(4, len(render_imgs))
    fig, axes = plt.subplots(2, n_show, figsize=(5 * n_show, 10))
    if n_show == 1:
        axes = axes.reshape(2, 1)

    for i in range(n_show):
        render_img = Image.open(render_imgs[i])
        axes[0, i].imshow(render_img)
        axes[0, i].set_title(f"Render {i}")
        axes[0, i].axis("off")

        if i < len(gt_imgs):
            gt_img = Image.open(gt_imgs[i])
            axes[1, i].imshow(gt_img)
            axes[1, i].set_title(f"GT {i}")
            axes[1, i].axis("off")

    plt.suptitle(f"3DGS + Custom BA - {SCENE_NAME} (1/{IMAGE_SCALE})", fontsize=16)
    plt.tight_layout()
    plt.show()
    print(f"✅ {len(render_imgs)}장 렌더링 이미지 중 {n_show}장 표시")